In [1]:
import pandas as pd
from pathlib import Path

# =========================
# Settings
# =========================
PROJECT_ROOT = Path.cwd().parent   # nếu notebook nằm trong experiments/
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_PATH = DATA_DIR / "Air_Quality_Processed.csv"


OUT_DIR = DATA_DIR / "station_split_24havg"

# split ratios (time-based)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Ratios must sum to 1.0"

ROLL_WIN = 24  # 24h average target

# =========================
# Load df_all
# =========================
if "df_all" not in globals():
    df_all = pd.read_csv(PROCESSED_PATH)
else:
    df_all = df_all.copy()

df_all["date"] = pd.to_datetime(df_all["date"])
df_all = df_all.sort_values(["Station_No", "date"]).reset_index(drop=True)

# =========================
# Create 24h-avg target 
# =========================
# Target mới: PM2.5_24h_avg = rolling mean 24 theo giờ, theo từng station
df_all["PM2.5_24h_avg"] = (
    df_all.groupby("Station_No")["PM2.5"]
    .transform(lambda s: s.rolling(window=ROLL_WIN, min_periods=ROLL_WIN).mean())
)

# bỏ các dòng chưa đủ 24h để có target
before = len(df_all)
df_all = df_all.dropna(subset=["PM2.5_24h_avg"]).reset_index(drop=True)
after = len(df_all)
print(f"Dropped {before-after} rows without 24h avg target. Remaining: {after}")

# =========================
# Create station_split
# =========================
OUT_DIR.mkdir(parents=True, exist_ok=True)

stations = sorted(df_all["Station_No"].dropna().unique().tolist())
summary_rows = []

MIN_ROWS = 500  

for st in stations:
    df_st = df_all[df_all["Station_No"] == st].sort_values("date").reset_index(drop=True)

    n = len(df_st)
    if n < MIN_ROWS:
        print(f"Skip station {st}: too few rows ({n})")
        continue

    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    n_test  = n - n_train - n_val

    train_df = df_st.iloc[:n_train].copy()
    val_df   = df_st.iloc[n_train:n_train+n_val].copy()
    test_df  = df_st.iloc[n_train+n_val:].copy()

    st_dir = OUT_DIR / f"station_{int(st)}"
    st_dir.mkdir(parents=True, exist_ok=True)

    train_path = st_dir / "train.csv"
    val_path   = st_dir / "val.csv"
    test_path  = st_dir / "test.csv"

    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path, index=False)
    test_df.to_csv(test_path, index=False)

    summary_rows.append({
        "Station_No": int(st),
        "n_total": n,
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        "date_min": df_st["date"].min(),
        "date_max": df_st["date"].max(),
    })

print(f"\nSaved station_split (24h avg target) to: {OUT_DIR}")
summary = pd.DataFrame(summary_rows).sort_values("Station_No").reset_index(drop=True)
summary


Dropped 138 rows without 24h avg target. Remaining: 69377

Saved station_split (24h avg target) to: c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\data\station_split_24havg


,Station_No,n_total,n_train,n_val,n_test,date_min,date_max
0,1,11566,8096,1734,1736,2021-02-24 20:00:00,2022-06-21 17:00:00
1,2,11566,8096,1734,1736,2021-02-24 20:00:00,2022-06-21 17:00:00
2,3,11566,8096,1734,1736,2021-02-24 20:00:00,2022-06-21 17:00:00
3,4,11566,8096,1734,1736,2021-02-24 20:00:00,2022-06-21 17:00:00
4,5,11566,8096,1734,1736,2021-02-24 20:00:00,2022-06-21 17:00:00
5,6,11547,8082,1732,1733,2021-02-25 15:00:00,2022-06-21 17:00:00
